# Apex Predictor — Deep Learning: Graph Neural Network

Quarto e ultimo esperimento. Un grafo per gara (nodi = piloti, archi = relazione compagno di squadra), classificazione di nodo con propagazione di informazione lungo gli archi.
La GNN può imparare come combinare le informazioni del compagno di squadra, oltre alla singola feature teammate_position_gap già presente.

Confronto: XGBoost F1 0.717, TabNet F1 0.711, LSTM F1 0.678, MLP F1 0.657.

In [1]:
import sys
sys.path.append("..")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv
from torch_geometric.loader import DataLoader as GraphDataLoader
import numpy as np

from src.data_loading import load_raw_data, build_working_dataset
from src.features import build_all_features
from src.train import FEATURE_COL, temporal_split
from src.dl_common import build_race_graphs, find_best_threshold_np

torch.manual_seed(42)

data = load_raw_data()
df = build_working_dataset(data["races"], data["results"], min_year=2004)
df = build_all_features(
    df, data["circuits"], data["drivers"], data["constructors"],
    data["driver_standings"], data["constructor_standings"], data["qualifying"]
)

train_df, test_df = temporal_split(df)

feat_mean = train_df[FEATURE_COL].mean()
feat_std = train_df[FEATURE_COL].std().replace(0, 1)

train_graphs = build_race_graphs(train_df, FEATURE_COL, feat_mean, feat_std)
test_graphs = build_race_graphs(test_df, FEATURE_COL, feat_mean, feat_std)

print(f"Grafi di training: {len(train_graphs)} (gare)")
print(f"Grafi di test: {len(test_graphs)} (gare)")
print(f"Nodi nel primo grafo di training: {train_graphs[0].num_nodes}, archi: {train_graphs[0].num_edges}")

train_loader = GraphDataLoader(train_graphs, batch_size=8, shuffle=True)
test_loader = GraphDataLoader(test_graphs, batch_size=32, shuffle=False)

/home/daniele/progetti/apex-predictor/venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


Grafi di training: 411 (gare)
Grafi di test: 37 (gare)
Nodi nel primo grafo di training: 18, archi: 16


In [2]:
class RaceGNN(nn.Module):
    """
    GraphSAGE a due layer: ciascun nodo (pilota) aggrega informazione
    dai nodi vicini (compagni di squadra) per aggiornare la propria
    rappresentazione, poi un classificatore finale decide podio sì/no
    per ciascun nodo.
    """
    def __init__(self, num_features, hidden_dim=32):
        super().__init__()
        # SAGEConv: ad ogni layer, la rappresentazione di un nodo viene aggiornata combinando la propria con la media delle rappresentazioni dei nodi collegati
        # con due layer, l'informazione può propagarsi fino a 2 salti di distanza nel grafo
        self.conv1 = SAGEConv(num_features, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = self.dropout(x)
        x = F.relu(self.conv2(x, edge_index))
        x = self.dropout(x)
        return self.classifier(x).squeeze(-1)


model = RaceGNN(num_features=len(FEATURE_COL))
print(model)

RaceGNN(
  (conv1): SAGEConv(14, 32, aggr=mean)
  (conv2): SAGEConv(32, 32, aggr=mean)
  (classifier): Linear(in_features=32, out_features=1, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)


In [3]:
all_y_train = torch.cat([g.y for g in train_graphs])
n_pos = all_y_train.sum()
n_neg = len(all_y_train) - n_pos
pos_weight = torch.tensor(n_neg / n_pos)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)


def evaluate_loss(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            logits = model(batch.x, batch.edge_index)
            total_loss += criterion(logits, batch.y).item()
    return total_loss / len(loader)


N_EPOCHS = 100
patience = 15
best_test_loss = float("inf")
epochs_without_improvement = 0
best_model_state = None

for epoch in range(N_EPOCHS):
    model.train()
    epoch_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        logits = model(batch.x, batch.edge_index)
        loss = criterion(logits, batch.y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    test_loss = evaluate_loss(model, test_loader, criterion)

    if test_loss < best_test_loss:
        best_test_loss = test_loss
        epochs_without_improvement = 0
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        epochs_without_improvement += 1

    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/{N_EPOCHS} — train loss: {epoch_loss/len(train_loader):.4f}, test loss: {test_loss:.4f}")

    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping all'epoch {epoch+1}")
        break

model.load_state_dict(best_model_state)
print(f"Miglior test loss: {best_test_loss:.4f}")

/tmp/ipykernel_4837/409691488.py:4: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  pos_weight = torch.tensor(n_neg / n_pos)


Epoch 5/100 — train loss: 0.5916, test loss: 0.5889
Epoch 10/100 — train loss: 0.5747, test loss: 0.5661
Epoch 15/100 — train loss: 0.5549, test loss: 0.5527
Epoch 20/100 — train loss: 0.5349, test loss: 0.5451
Epoch 25/100 — train loss: 0.5337, test loss: 0.5535
Epoch 30/100 — train loss: 0.5212, test loss: 0.5530
Epoch 35/100 — train loss: 0.5139, test loss: 0.5520

Early stopping all'epoch 37
Miglior test loss: 0.5417


In [4]:
model.eval()
all_probas, all_targets = [], []
with torch.no_grad():
    for batch in test_loader:
        logits = model(batch.x, batch.edge_index)
        probas = torch.sigmoid(logits)
        all_probas.append(probas.numpy())
        all_targets.append(batch.y.numpy())

y_proba = np.concatenate(all_probas)
y_true = np.concatenate(all_targets)

best = find_best_threshold_np(y_true, y_proba)
print(f"GNN (GraphSAGE) — soglia {best['threshold']:.2f}: "
      f"precision {best['precision']:.3f}, recall {best['recall']:.3f}, F1 {best['f1']:.3f}")
print(f"\nConfronto — XGBoost: F1 0.717 | TabNet: F1 0.711 | LSTM: F1 0.678 | MLP: F1 0.657")

GNN (GraphSAGE) — soglia 0.75: precision 0.634, recall 0.703, F1 0.667

Confronto — XGBoost: F1 0.717 | TabNet: F1 0.711 | LSTM: F1 0.678 | MLP: F1 0.657
